In [43]:
import pandas as pd
import numpy as np
import default_risk.config as cfg

column_order_reference="days_instalment"
installments_payment_df = pd.read_parquet(cfg.CLEANS_DIR / "installments_payments_train-cleaned.parquet")
installments_payment_df.sort_values(["id_prev",column_order_reference,"days_instalment"],inplace=True)


# aux functions
def get_time_window_metrics(days_since, installments_payment_df) :    
    last_year= installments_payment_df[installments_payment_df["days_instalment"] > days_since]

    days_since = days_since * -1

    last_year_agg= last_year.groupby("id_curr").agg({
        "amt_instalment": ["max", "min","mean","sum"],
        "amt_payment": ["max", "min","mean","sum","std"],
        "days_of_delinquency":["mean","sum"],
        "days_in_advance":["mean","sum","max"],
        "diff_expected_received": ["max", "min", "mean", "sum"],
        "is_delinquency" : ["mean","sum"],
        "is_underpayment" : ["mean"],
        "extra_instalament":["sum","mean"],
        })


    last_year_agg.columns = [
    f"last_{days_since}_instalments_{col[0]}_{col[1]}"
    for col in last_year_agg.columns
    ]

    last_year_agg= last_year_agg.reset_index()

    last_year_agg[f"last_{days_since}_instalments_completion_ratio"] = np.where( last_year_agg[f"last_{days_since}_instalments_amt_instalment_sum"] > 0, last_year_agg[f"last_{days_since}_instalments_amt_payment_sum"] / last_year_agg[f"last_{days_since}_instalments_amt_instalment_sum"], 1.0 ) 

    last_year_agg.to_parquet(cfg.PROCESSED_DIR / "installments_payments_last_year_metrics.parquet")
    return last_year_agg



In [44]:
installments_payment_df.head()


,id_prev,id_curr,days_decision,num_instalment_version,initial_version,num_instalment_number,days_instalment,days_entry_payment,days_entry_payment_is_missing,amt_instalment,amt_payment
0,1000001,158271,-299.0,1.0,1.0,1,-268.0,-294.0,0,6404.310,6404.310
1,1000001,158271,-299.0,2.0,1.0,2,-238.0,-244.0,0,62039.115,62039.115
2,1000003,252457,-124.0,1.0,1.0,1,-94.0,-108.0,0,4951.350,4951.350
3,1000003,252457,-124.0,1.0,1.0,2,-64.0,-81.0,0,4951.350,4951.350
4,1000003,252457,-124.0,1.0,1.0,3,-34.0,-49.0,0,4951.350,4951.350


In [45]:
#we starting catching this because we are probably cutting some parts of the temporal sequence
installments_payment_df["raw_size_serie"]= installments_payment_df.groupby("id_prev").transform("size")
installments_payment_df["amount_of_versions_in_sequence"] = installments_payment_df.groupby("id_prev")["num_instalment_version"].transform("nunique")

#we gonna use the knowlege recolected in the EDA. 
#For more details look eda_installments_payments.ipynb decisions summary 1#

#creating auxiliar columns
installments_payment_df["next_payment_value"] = installments_payment_df.groupby("id_prev")["days_entry_payment"].shift(-1)
nan_amount= installments_payment_df.groupby("id_prev")["days_entry_payment_is_missing"].transform("sum")

#defining the mask to separate the differents cases of missings values 
have_missing_entry_payment_mask= (installments_payment_df["next_payment_value"].isna())
not_a_deadtail_mask= (installments_payment_df["days_entry_payment"].isna() ) & ( installments_payment_df["next_payment_value"].notna())

#we want to catch the cases of  dead-tail so we starting filtering that cases with nans but that are not dead tails
non_dead_tail_nans= installments_payment_df[not_a_deadtail_mask]
ids_with_nulls_that_are_non_deadtails= non_dead_tail_nans["id_prev"].unique()
excluding_nans_non_deadtails_mask= ~(installments_payment_df["id_prev"].isin(ids_with_nulls_that_are_non_deadtails))
series_without_problematic_nans = installments_payment_df[excluding_nans_non_deadtails_mask]

#now this ID are series where the nans are deadtails, so count nans in "days_entry_payment" o "amt_payment" is calculate
#the lenght of the deadtail and we save that value in a new column
ids_with_deadtails= series_without_problematic_nans[series_without_problematic_nans["days_entry_payment"].isna()]["id_prev"].unique()
installments_payment_df["dead_tail_length"]= np.where(installments_payment_df["id_prev"].isin(ids_with_deadtails),  nan_amount, 0)

#also in the decision summary of the EDA we define a criteria to incomplete series 
starting_instalment_number= installments_payment_df.groupby("id_prev")["num_instalment_number"].transform("first")
starting_date= installments_payment_df.groupby("id_prev")["days_instalment"].transform("first")

installments_payment_df["is_potentially_incomplete_sequence"] = ((starting_instalment_number >  1)  & (starting_date < -2890 ))

potentially_on_going_id = installments_payment_df[installments_payment_df["days_instalment"] > (- 33)]["id_prev"].unique()
installments_payment_df["potentially_on_going"]= installments_payment_df["id_prev"].isin(potentially_on_going_id)   



installments_payment_df["is_underpayment"]= (installments_payment_df["amt_instalment"] >  installments_payment_df["amt_payment"]) & (installments_payment_df["amt_payment"] != 0)
rows_with_underpayment= installments_payment_df [installments_payment_df["is_underpayment"] == True]
installments_payment_df["days_of_underpayment"] = np.where(installments_payment_df["is_underpayment"], installments_payment_df["days_instalment"], np.nan)

installments_payment_df["diff_expected_received"]= installments_payment_df["amt_instalment"] -  installments_payment_df["amt_payment"]
installments_payment_df["diff_deadline_factical_payment"]= installments_payment_df["days_entry_payment"] - installments_payment_df["days_instalment"] 
installments_payment_df["days_in_advance"] = installments_payment_df["days_instalment"] - installments_payment_df["days_entry_payment"]
installments_payment_df["days_of_delinquency"]= installments_payment_df["diff_deadline_factical_payment"].clip(lower=0)
installments_payment_df["days_in_advance"]= installments_payment_df["days_in_advance"].clip(lower=0)
installments_payment_df["is_delinquency"] = installments_payment_df["days_of_delinquency"] > 0 



next_installment_number = installments_payment_df.groupby("id_prev")["num_instalment_number"].shift(-1)
next_version_number = installments_payment_df.groupby("id_prev")["num_instalment_version"].shift(-1)
repeated_installment_mask= (installments_payment_df ["num_instalment_number"] == next_installment_number)
underpayment_mask= (installments_payment_df[ "amt_payment" ] < installments_payment_df ["amt_instalment"])
full_payment_mask= installments_payment_df[ "amt_payment" ] == installments_payment_df ["amt_instalment"] 
installments_payment_df[ "repeated_for_underpayment" ] = (repeated_installment_mask) & (underpayment_mask)
#installments_payment_df[ "repeated_for_reschedule" ] = (repeated_installment_mask) & (full_payment_mask)
#installments_payment_df["repeated_for_payment_in_advance"] = (repeated_installment_mask) & (installments_payment_df[ "amt_payment" ] == 0) & (installments_payment_df["days_of_delinquency"] == 0)
installments_payment_df["log_amt_instalment"]= np.log1p(installments_payment_df ["amt_instalment"] )
installments_payment_df["log_amt_payment"]= np.log1p(installments_payment_df ["amt_payment"] )

installments_payment_df["extra_instalament"] = (installments_payment_df["amount_of_versions_in_sequence"] > 1) & (installments_payment_df["raw_size_serie"] <95)  & (installments_payment_df ["num_instalment_number"] >= 100)
interesting_cases= installments_payment_df[installments_payment_df["extra_instalament"] == True ]

installments_payment_df["potentially_on_going"]= installments_payment_df["potentially_on_going"].astype(int)




In [46]:
agg_metrics_df= installments_payment_df.groupby("id_prev").agg({

    #static values calculated for the entire squenece
    "raw_size_serie" : ["first"],
    "dead_tail_length" : ["first"],
    "potentially_on_going" : ["first"],
    "amount_of_versions_in_sequence" : ["first"],

    #for log transformated we want to catch the mean and the std (avoiding the impact of the heavy tail from this columns)
    "log_amt_instalment": ["mean","std"],   
    "log_amt_payment": ["mean","std"],

    #natural scale
    "amt_instalment": ["max", "min","median","sum"], #, "count"
    "amt_payment": ["max", "min","median","sum"],
    "extra_instalament":["sum","mean"],
   
    #computed_differences
    "diff_expected_received": ["max", "min", "median", "sum"],
    
    #categoricals
    "repeated_for_underpayment": ["mean","sum"],
    "is_delinquency" : ["mean","sum"],

    #counters
    "days_of_delinquency":["mean","max","sum"],
    "days_in_advance":["mean","max","sum"],
    "days_of_underpayment" : ["max"]
})

agg_metrics_df.columns = [
    f"instalments_{col[0]}" if col[1] == "first" else f"instalments_{col[0]}_{col[1]}"
    for col in agg_metrics_df.columns
]

agg_metrics_df = agg_metrics_df.reset_index()
#if the debt is 0 or negative we assume competitud (1)
agg_metrics_df["instalments_completion_ratio"] = np.where( agg_metrics_df["instalments_amt_instalment_sum"] > 0, agg_metrics_df["instalments_amt_payment_sum"] / agg_metrics_df["instalments_amt_instalment_sum"], 1.0 )     
agg_metrics_df.to_parquet(cfg.PROCESSED_DIR / "installments_payments.train-processed-2.parquet")


In [47]:
agg_metrics_last_three_months= get_time_window_metrics(-90, installments_payment_df)
agg_metrics_last_year = get_time_window_metrics(-365, installments_payment_df)
last_two_years = get_time_window_metrics(-720, installments_payment_df)

temporal_windows_df = agg_metrics_last_year.merge(
    agg_metrics_last_three_months, 
    on="id_curr", 
    how="outer",  
)

temporal_windows_df = temporal_windows_df.merge(
    last_two_years, 
    on="id_curr", 
    how="outer",  
)


temporal_windows_df["payment_trend"] = np.where(temporal_windows_df["last_90_instalments_completion_ratio"] != 0 , temporal_windows_df["last_365_instalments_completion_ratio"] / temporal_windows_df["last_90_instalments_completion_ratio"],np.nan)
temporal_windows_df["delincuency_trend"] = np.where(temporal_windows_df["last_365_instalments_days_of_delinquency_mean"] != 0 , temporal_windows_df["last_720_instalments_days_of_delinquency_mean"] / temporal_windows_df["last_365_instalments_days_of_delinquency_mean"],np.nan)
temporal_windows_df["underpayment_trend"] = np.where(temporal_windows_df["last_365_instalments_is_underpayment_mean"] != 0 , temporal_windows_df["last_720_instalments_is_underpayment_mean"] / temporal_windows_df["last_365_instalments_is_underpayment_mean"],np.nan)



#temporal_windows_df["delincuency_trend"] = temporal_windows_df["last_365_instalments_days_of_delinquency_mean"] - temporal_windows_df["last_90_instalments_days_of_delinquency_mean"] 
#temporal_windows_df["underpayment_trend"] = temporal_windows_df["last_365_instalments_is_underpayment_mean"] - temporal_windows_df["last_90_instalments_is_underpayment_mean"]

temporal_windows_df_to_save=pd.DataFrame()

temporal_windows_df_to_save["last_720_instalments_days_of_delinquency_mean"] =  temporal_windows_df["last_720_instalments_days_of_delinquency_mean"] 
temporal_windows_df_to_save["last_365_instalments_days_of_delinquency_sum"]= temporal_windows_df["last_365_instalments_days_of_delinquency_sum"]
temporal_windows_df_to_save["last_365_instalments_days_of_delinquency_mean"]= temporal_windows_df["last_365_instalments_days_of_delinquency_mean"]
temporal_windows_df_to_save["last_365_instalments_extra_instalament_mean"]= temporal_windows_df["last_365_instalments_extra_instalament_mean"]
temporal_windows_df_to_save["last_365_instalments_amt_payment_min"]= temporal_windows_df["last_365_instalments_amt_payment_min"]
temporal_windows_df_to_save["last_90_instalments_amt_instalment_min"]= temporal_windows_df["last_90_instalments_amt_instalment_min"]
temporal_windows_df_to_save["last_365_instalments_is_delinquency_mean"]= temporal_windows_df["last_365_instalments_is_delinquency_mean"]
temporal_windows_df_to_save["last_720_instalments_completion_ratio"]= temporal_windows_df["last_720_instalments_completion_ratio"]
temporal_windows_df_to_save["last_365_instalments_completion_ratio"]= temporal_windows_df["last_365_instalments_completion_ratio"] 
temporal_windows_df_to_save["last_90_instalments_completion_ratio"]= temporal_windows_df["last_90_instalments_completion_ratio"] 
temporal_windows_df_to_save["last_90_instalments_amt_instalment_min"]= temporal_windows_df["last_90_instalments_amt_instalment_min"]

temporal_windows_df_to_save["payment_trend"]= temporal_windows_df["payment_trend"]
temporal_windows_df_to_save["delincuency_trend"]= temporal_windows_df["delincuency_trend"]




temporal_windows_df_to_save["id_curr"]= temporal_windows_df["id_curr"]
temporal_windows_df_to_save.to_parquet(cfg.PROCESSED_DIR / "time_window_instalments.parquet")


